In [19]:
# Imports
import sys
import os
sys.path.append(os.path.abspath('../..'))


from controller.marl.main import setup
from controller.marl.core.config import Config
from controller.marl.runners.sim_runner import run_sim
from controller.marl.models.aim import AIM

from project_paths import PROJECT_ROOT


import torch
from controller.marl.core.datasets import FilteredObsData
from torch.utils.data import DataLoader


from notebooks.plt_style import set_style
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from sklearn.cluster import MiniBatchKMeans


In [20]:
set_style()

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [22]:
config = Config.from_yaml(PROJECT_ROOT / "configs")

In [23]:
system, config = setup(config, device)

Loaded language: C:\Users\green\OneDrive\Desktop\Cambridge\Part II\Dissertation\Project\results\languages\2026-05-07_09-50-55


In [24]:
run_sim(system, config, device, 5, collect_obs_file="./temp.csv", optimal=True)

Running Simulation: 100%|██████████| 5/5 [00:00<00:00, 12.00it/s]


Exporting Buffer to ./temp.csv...
Done!


np.float32(0.23528583)

In [25]:
obs_logs_file = "./temp.csv"

GO = system["sim"].get_global_obs_dim()
mask = torch.tensor(system["sim"].get_agent_external_obs_mask(0), dtype=torch.bool, device=device)
dataset = FilteredObsData(obs_logs_file, system["act_shape"][0], GO, mask, device)

dataloader = DataLoader(dataset, batch_size=config.aim_training.aim_batch_size, shuffle=True)

In [26]:

obs_external_mask = torch.tensor(system["sim"].get_agent_external_obs_mask(0), dtype=torch.bool, device=device)
obs_mask = torch.tensor(system["sim"].get_agent_obs_mask(0), dtype=torch.bool, device=device)[obs_external_mask]

OBS_DIM = obs_external_mask.sum().int().item()

num_training_steps = config.aim_training.ae_epochs * len(dataloader)


aim = AIM(OBS_DIM, config.comms, config.aim_training, obs_mask=obs_mask, num_training_steps=num_training_steps).to(device)

In [27]:
aim.encoder.eval()
all_latents = []

print(f"Extracting latents from {len(dataloader.dataset)} samples...")

with torch.no_grad():
    for batch in tqdm(dataloader):
        obs = batch[0].to(device)
        
        latent = aim.encoder.get_continuous_latent(obs)[0]
        
        all_latents.append(latent.cpu().numpy())

residuals = np.concatenate(all_latents, axis=0)

Extracting latents from 19199 samples...


100%|██████████| 75/75 [00:12<00:00,  5.83it/s]


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from project_paths import FIGURES_DIR

inertias = []
vocab_sizes = range(2, 65, 2)

flat_residuals = residuals.reshape(-1, residuals.shape[-1])

for k in tqdm(vocab_sizes):
    kmeans = MiniBatchKMeans(n_clusters=k, batch_size=1024, n_init="auto")
    kmeans.fit(flat_residuals)
    inertias.append(kmeans.inertia_)
    
plt.figure(figsize=(8, 5))
plt.plot(vocab_sizes, inertias, 'bx-')
plt.xlabel('Vocabulary Size (k)')
plt.ylabel('Inertia (Quantisation Error)')
plt.title('Elbow Method for Optimal Vocab Size')
plt.savefig(FIGURES_DIR / "elbow-curve.png", dpi=900)
plt.close()

100%|██████████| 32/32 [00:06<00:00,  5.02it/s]
